In [2]:
import pandas as pd
import re

# Load the nginx log file
def parse_nginx_log(log_file):
    logs = []
    regex = r"^(\S+) - - \[(.*?)\] \"(\S+) (.*?) (HTTP\/\d\.\d)\" (\d{3}) (\d+) \"(.*?)\" \"(.*?)\""
    
    with open(log_file, 'r') as f:
        for line in f:
            match = re.match(regex, line)
            if match:
                logs.append(match.groups())

    columns = ["client_ip", "timestamp", "method", "request", "protocol", "status", "response_size", "referrer", "user_agent"]
    return pd.DataFrame(logs, columns=columns)

# Labeling rules
def label_data(df):
    labels = []
    
    for _, row in df.iterrows():
        if int(row['status']) == 404 and row['request'].endswith('/wp-login.php'):
            labels.append("Brute Force")
        elif re.search(r".*(UNION|SELECT|INSERT|DELETE|UPDATE).*", row['request'], re.IGNORECASE):
            labels.append("SQL Injection")
        elif re.search(r".*(<script>|%3Cscript%3E).*", row['request'], re.IGNORECASE):
            labels.append("XSS Attack")
        elif '../' in row['request'] or '..%2F' in row['request']:
            labels.append("Directory Traversal")
        else:
            labels.append("Normal")
    
    df['label'] = labels
    return df

# Main pipeline
log_file = '/database tubes/dataset.txt'  # Replace with your log file path

# Step 1: Parse Logs
raw_data = parse_nginx_log(log_file)

# Step 2: Label Data
labeled_data = label_data(raw_data)

# Save to CSV
output_file = '/database tubes/labeled_logs.csv'
labeled_data.to_csv(output_file, index=False)
output_file


FileNotFoundError: [Errno 2] No such file or directory: '/database tubes/dataset.txt'